# 05. External data

Downloads the two public datasets the analysis uses into `data/` (committed, so the other notebooks run offline):

* **Surface weather** around the origin: ASOS/METAR observations at XMR (Cape Canaveral Skid Strip), TTS
  (Shuttle Landing Facility) and COF (Patrick Space Force Base), 2026-05-28 22:00 to 2026-05-29 04:00 UTC, from
  the Iowa Environmental Mesonet. Used by `newglenn.yaml` (`weather`) and notebooks 22-34.
* **2026 Florida launches**: launches from Cape Canaveral SFS and KSC (Launch Library 2 locations 12 and 27)
  between 2026-01-01 and 2026-09-15, from the public Launch Library 2 API. Used by notebook 32.

With `REFRESH = False` existing files are kept (so a rerun does not depend on the services). With
`REFRESH = True` both are downloaded again and compared with the committed copies. The Launch Library free API
allows about 15 requests per hour.

In [ ]:
import io, json, urllib.request
import pandas as pd
import newglenn_config as cfg

REFRESH = False
DATA = cfg.PAPER_DIR / 'data'
DATA.mkdir(exist_ok=True)
UA = {'User-Agent': 'ksc-seismo-acoustic-research'}

def get(url):
    with urllib.request.urlopen(urllib.request.Request(url, headers=UA), timeout=60) as r:
        return r.read()

## Weather (Iowa Environmental Mesonet ASOS archive)

In [ ]:
ASOS = DATA / 'iem_asos_XMR_TTS_COF_20260528T22_20260529T04.csv'
ASOS_URL = ('https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py?station=XMR&station=TTS&station=COF'
            '&data=tmpf&data=dwpf&data=relh&data=drct&data=sknt&data=mslp&data=alti'
            '&year1=2026&month1=5&day1=28&hour1=22&year2=2026&month2=5&day2=29&hour2=4'
            '&tz=Etc/UTC&format=onlycomma&latlon=yes&report_type=3&report_type=4')
if REFRESH or not ASOS.exists():
    new = get(ASOS_URL)
    if ASOS.exists() and ASOS.read_bytes() != new:
        print('NOTE: the downloaded weather differs from the committed copy')
    ASOS.write_bytes(new)
    print('downloaded', ASOS.name)
w = pd.read_csv(ASOS)
print(len(w), 'observations;', w.station.value_counts().to_dict())
display(w.head())

## 2026 Florida launches (Launch Library 2)

In [ ]:
LAUNCHES = DATA / 'launches_florida_2026_launchlibrary.csv'
LL_URL = ('https://ll.thespacedevs.com/2.3.0/launches/previous/?net__gte=2026-01-01T00:00:00Z'
          '&net__lte=2026-09-15T00:00:00Z&location__ids=12,27&limit=100&mode=list')
if REFRESH or not LAUNCHES.exists():
    d = json.loads(get(LL_URL))
    rows = [dict(net=r['net'], name=r['name'], status=(r.get('status') or {}).get('abbrev', ''), slug=r.get('slug', ''))
            for r in d['results']]
    new = pd.DataFrame(rows).sort_values('net')
    if d.get('next'):
        print('WARNING: more than 100 launches; only the first page was read')
    if LAUNCHES.exists():
        old = pd.read_csv(LAUNCHES)
        if not old.equals(new.reset_index(drop=True)):
            print('NOTE: the downloaded launch list differs from the committed copy')
    new.to_csv(LAUNCHES, index=False)
    print('downloaded', LAUNCHES.name)
L = pd.read_csv(LAUNCHES)
print(len(L), 'launches,', L.net.min(), 'to', L.net.max())
display(L.tail())